# TimeOmni-VL 用户侧日前电价预测 — 全流程 Notebook

本 Notebook 复现 TimeOmni-VL 的核心数据流：
1. 数据加载与对齐
2. 数据清洗与样本生成
3. Bi-TSI 双向转换（TS2I / I2TS）
4. 理解任务与生成 CoT 构造
5. Mock Backbone 训练
6. 推理与评估

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 确保项目根目录在 PYTHONPATH 中
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 切换工作目录到项目根目录
os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')
from timeomni_vl.config import ConfigManager
from timeomni_vl.data.pipeline import build_data_pipeline
from timeomni_vl.bitsi import RobustFidelityNormalizer, TS2IConverter, I2TSConverter, BiTSIValidator
from timeomni_vl.models import build_adapter
from timeomni_vl.tasks.understanding import UnderstandingTaskGenerator
from timeomni_vl.tasks.generation import GenerationTaskGenerator, GenerationEvaluator
from timeomni_vl.training.dataset import TSUMMDatasetBuilder
from timeomni_vl.inference.understanding import UnderstandingInferencer
from timeomni_vl.inference.generation import GenerationInferencer

print('Imports OK')

## 1. 加载配置

In [ ]:
cfg = ConfigManager('configs/train.yaml')
data_cfg = cfg.get_data_config()
bitsi_cfg = cfg.get_bitsi_config()
model_cfg = cfg.get_model_config()
training_cfg = cfg.get_training_config()

print(f"backbone: {model_cfg.backbone}")
print(f"target variable: {data_cfg.target_variable}")
print(f"context days: {data_cfg.context_days}, forecast days: {data_cfg.forecast_days}")
print(f"image size: {bitsi_cfg.image_size}")

## 2. 数据加载、对齐、清洗与采样

In [ ]:
pipeline = build_data_pipeline(data_cfg)
samples = pipeline.run()

print(f"\nTrain samples: {len(samples['train'])}")
print(f"Val samples: {len(samples['val'])}")
print(f"Test samples: {len(samples['test'])}")

sample = samples['train'][0]
print(f"\nSample context shape: {sample.context.shape}")
print(f"Sample target shape: {sample.target.shape}")
print(f"Sample task: {sample.task}")

## 3. Bi-TSI：时间序列 ⇄ 图像

In [ ]:
rfn = RobustFidelityNormalizer(
    alpha=bitsi_cfg.alpha,
    c_mad=bitsi_cfg.c_mad,
    kappa=bitsi_cfg.kappa
)
ts2i = TS2IConverter(
    frequency=data_cfg.frequency,
    image_size=448,  # 使用较小尺寸便于 Notebook 快速验证
    color_map=bitsi_cfg.color_map
)
i2ts = I2TSConverter(
    frequency=data_cfg.frequency,
    image_size=448
)

ctx_norm, stats = rfn.fit_transform(sample.context)
src_image = ts2i.convert(ctx_norm)

print(f"Source TS-image size: {src_image.size}")
src_image

### 3.1 Round-trip 验证

In [ ]:
validator = BiTSIValidator(rfn, ts2i, i2ts)
metrics = validator.validate(sample.context[:96*3])  # 用 3 天数据验证
print(f"Round-trip metrics: {metrics}")

## 4. 理解任务生成

In [ ]:
qa_gen = UnderstandingTaskGenerator(
    frequency=data_cfg.frequency,
    image_size=448
)
qas = qa_gen.generate_all(sample)

for qa in qas:
    print(f"\n{qa['task_id']}: {qa['question']}")
    print(f"  CoT: {qa['cot'][:80]}...")
    print(f"  Answer: {qa['answer']}")

## 5. 生成 CoT 与训练数据集构造

In [ ]:
gen_gen = GenerationTaskGenerator(
    frequency=data_cfg.frequency,
    context_days=data_cfg.context_days,
    forecast_days=data_cfg.forecast_days
)
cot = gen_gen.generate_cot(qas)
print('Generated CoT:')
print(cot)

dataset_builder = TSUMMDatasetBuilder(data_cfg, bitsi_cfg)
train_dataset = dataset_builder.build(samples['train'][:3])
print(f"\nBuilt {len(train_dataset)} training items")
print(f"Keys: {list(train_dataset[0].keys())}")

## 6. 加载 Backbone 并训练（Mock）

In [ ]:
adapter = build_adapter(model_cfg.backbone)
adapter.load(model_cfg.model_path, model_cfg.device)
print(f"Adapter loaded: {type(adapter).__name__}")

# 模拟训练循环
history = []
for epoch in range(1, training_cfg.num_epochs + 1):
    total_loss = 0.0
    for item in train_dataset:
        metrics = adapter.train_step(item)
        total_loss += metrics['loss_total']
    avg_loss = total_loss / len(train_dataset)
    history.append(avg_loss)
    print(f"Epoch {epoch}/{training_cfg.num_epochs}, loss: {avg_loss:.4f}")

## 7. 推理：理解与生成

In [ ]:
# 7.1 理解推理
understand_inferencer = UnderstandingInferencer(adapter)
understand_result = understand_inferencer.infer(
    src_image,
    question='How many variables are encoded in this TS-image?'
)
print('Understanding result:')
print(understand_result)

In [ ]:
# 7.2 生成推理
gen_inferencer = GenerationInferencer(adapter, i2ts, use_cot=True)
forecast_result = gen_inferencer.forecast(
    source_image=src_image,
    instruction='Predict the next day electricity price.',
    n_vars=sample.context.shape[1],
    target_length=data_cfg.forecast_days * data_cfg.points_per_day,
    stats=stats
)

print(f"CoT: {forecast_result['cot'][:100]}...")
print(f"Predicted values shape: {forecast_result['predicted_values'].shape}")
print(f"Predicted values range: [{forecast_result['predicted_values'].min():.2f}, {forecast_result['predicted_values'].max():.2f}]")

## 8. 评估与可视化

In [ ]:
# 提取目标变量索引
target_idx = 0  # 默认第一个变量为目标
for i, name in enumerate(train_dataset[0].get('variable_names', [])):
    if data_cfg.target_variable in name:
        target_idx = i
        break

pred = forecast_result['predicted_values']
if pred.ndim > 1:
    pred = pred[:, target_idx]
ref = sample.target[:len(pred), target_idx]

evaluator = GenerationEvaluator(metrics=['mae', 'rmse', 'nmape', 'direction'])
eval_metrics = evaluator.evaluate(pred, ref)
print('Evaluation metrics:')
print(eval_metrics)

plt.figure(figsize=(12, 4))
plt.plot(ref, label='Ground Truth', marker='o', markersize=3)
plt.plot(pred, label='Prediction', marker='x', markersize=3)
plt.title('Day-ahead Electricity Price Forecast')
plt.xlabel('Time Point (15-min)')
plt.ylabel('Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. 结果说明

- 当前使用 `MockAdapter`，理解输出和生成图像均为占位结果。
- 真实训练需接入 `Bagel-7B` 或 `Janus-Pro` 等 UMM backbone。
- 由于目标变量仅 25 天有效数据，训练样本稀少，真实训练时建议增加数据增强与滚动窗口。